In [1]:
using LowLevelFEM, LinearAlgebra

openGeometry("I-szelveny-2.geo")

In [2]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u)
U.non * U.pdim

902805

In [3]:
cs = CoordinateSystem([-1, 0, 0], [0, -1, 0])
Q = rotateNodes(U, "left", cs)
cs1 = BoundaryCondition("left", ux=0, uy=0, uz=0)
dofs = constrainedDoFs(U, [cs1])
for i in dofs
    Q.T[i, i] = -1
end

In [4]:
K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U), Ω="body")
K = Q' * K * Q

sparse([1, 2, 3, 115, 116, 117, 172, 173, 174, 1006  …  848445, 897604, 897605, 897606, 900346, 900347, 900348, 902803, 902804, 902805], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  902805, 902805, 902805, 902805, 902805, 902805, 902805, 902805, 902805, 902805], [204221.67825409558, 97603.31422229913, 68916.4136573893, 13044.233123468655, -4218.96210586296, -6151.263185232569, -61712.19663260604, 2797.9482778608904, 13490.013033078108, -9762.21126466965  …  -238133.27599437576, -316804.0884497837, -202083.35993243754, -130394.07204027659, 32865.39171450368, 20612.037814847514, -211804.7447597651, 853541.8626430527, -1.0292092451879622e6, 3.3440682855955376e6], 902805, 902805)

In [5]:
fx = ScalarField(U, "right", (x,y,z)->-y/1000)
fy = ScalarField(U, "right", (x,y,z)->(x-100)/1000)

elementwise ScalarField
[[-0.15; -0.145; … ; -0.14605046966569096; -0.14855046966569097;;], [-0.15; -0.15; … ; -0.14855046966569097; -0.14855046966569097;;], [-0.055; -0.05; … ; -0.05164362916539756; -0.05414362916539756;;], [-0.05; -0.05; … ; -0.05164362916539756; -0.05164362916539756;;], [-0.15; -0.15; … ; -0.14835637083460246; -0.14835637083460246;;], [-0.145; -0.15; … ; -0.14835637083460246; -0.14585637083460246;;], [-0.105; -0.11; … ; -0.10880649205366714; -0.10630649205366714;;], [-0.105; -0.105; … ; -0.10238220587689736; -0.10238220587689736;;], [-0.09976441175379472; -0.105; … ; -0.1025497632246402; -0.09993196910153757;;], [-0.1000995264492804; -0.105; … ; -0.1039542127754355; -0.1015039760000757;;]  …  [-0.085; -0.09; … ; -0.08874711708299494; -0.08624711708299494;;], [-0.08249380588907528; -0.085; … ; -0.08624711708299494; -0.0849940200275326;;], [-0.09; -0.09247159910686413; … ; -0.08998291663642702; -0.08874711708299494;;], [-0.053287258330795124; -0.05; … ; -0.05144953033

In [6]:
f = ∫(U ⋅ [fx, fy, 0], Γ="right")

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [7]:
showDoFResults(f, name="f");

In [8]:
bc = BoundaryCondition("right", ux=fx, uy=fy, uz=0)
bc2 = BoundaryCondition("left", uz=0)

BoundaryCondition("left", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uz => 0))

In [9]:
per = MPC(master="right", slave="left");

In [10]:
u = solveField(K, 0f, support=[bc, bc2], mpc=[per]);
u = Q * u;

In [11]:
showDoFResults(u, :uvec, name="displacement", factor=1);

In [12]:
u1 = nodesToElements(u, onPhysicalGroup="mid")
ε = (u1 ∘ ∇ + ∇ ∘ u1) / 2

E = mat.E
ν = mat.μ
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν / (1 - 2ν) * trace(ε) * I);

In [13]:
R = nodePositionVector(U)
R = nodesToElements(R, onPhysicalGroup="mid")

n2 = normalVector(U, "mid")

σ2 = elementsToElements(σ, onPhysicalGroup="mid")

τ2 = σ2 * n2;

In [14]:
showElementResults(τ2, name="shear stress");

In [15]:
showDoFResults(n2, name="normal vector");

In [16]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
